# Librerías

In [2]:
import gymnasium as gym # Importa la librería Gymnasium, que se usa para crear y manejar entornos de aprendizaje por refuerzo
import numpy as np # Importa NumPy, una librería para operaciones matemáticas y manejo eficiente de arreglos
from random import randint # Importa la función randint del módulo random para generar números enteros aleatorios
# Configurar NumPy para no usar notación científica al imprimir
np.set_printoptions(suppress=True)
import time

# Ambiente

[acrobot](https://gymnasium.farama.org/environments/classic_control/acrobot/)

![acrobot](https://gymnasium.farama.org/_images/acrobot.gif)

El entorno Acrobot se basa en el trabajo de Sutton en “Generalization in Reinforcement Learning: Successful Examples Using Sparse Coarse Coarse Coding” y en el libro de Sutton y Barto . El sistema consta de dos eslabones conectados linealmente para formar una cadena, con un extremo fijo. La articulación entre los dos eslabones es accionada. El objetivo es aplicar pares de torsión a la articulación accionada para elevar el extremo libre de la cadena lineal por encima de una altura determinada, partiendo de la posición inicial de suspensión hacia abajo.

Como se muestra en el GIF : dos eslabones azules conectados por dos articulaciones verdes. La articulación entre los dos eslabones se acciona.
- El objetivo es hacer girar el extremo libre del eslabón exterior hasta alcanzar la altura deseada (línea horizontal negra sobre el sistema) aplicando un par de torsión al actuador.

In [7]:
env= gym.make("Acrobot-v1")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<AcrobotEnv<Acrobot-v1>>>>>

**Objetivo del MDP**
- Lograr que la punta del brazo (extremo del segundo eslabón) alcance una altura determinada por encima de un punto de referencia (nivel de “goal height”).
- El agente debe controlar el **torque en la articulación central** para:
  - Balancear el brazo inferior y superior.
  - Generar suficiente energía cinética para que el extremo del brazo supere la altura objetivo.
  - Evitar movimientos demasiado bruscos que pierdan control.
- Requiere balancear entre:
  - Generar suficiente impulso para alcanzar la altura objetivo.
  - Evitar movimientos que reduzcan la energía del sistema.
  - Mantener estabilidad para no oscilar inútilmente y perder tiempo.

- Aplicar torque en la articulación central:  
  - **-1** → torque negativo  
  - **0** → sin torque  
  - **+1** → torque positivo

Recompensas
- **Negativa constante por paso**: penaliza tiempo prolongado (fuerza al agente a actuar rápido).  
- **Gran recompensa positiva**: si el extremo del brazo alcanza la altura objetivo.  
- **Penalización opcional por torque excesivo**: para limitar movimientos bruscos.

Fin del episodio
1. El extremo del brazo alcanza la altura objetivo.  
2. Se alcanza un número máximo de iteraciones (tiempo límite). 

**Espacio de Estados (State Space) - Acrobot**

La observación es un `ndarray` con forma `(6,)` que describe completamente el estado del Acrobot.

| Número | Observación                       | Mínimo       | Máximo       |
|--------|-----------------------------------|--------------|--------------|
| 0      | cos(θ₁)                           | -1           | 1            |
| 1      | sin(θ₁)                           | -1           | 1            |
| 2      | cos(θ₂)                           | -1           | 1            |
| 3      | sin(θ₂)                           | -1           | 1            |
| 4      | Velocidad angular θ̇₁             | -4π          | 4π           |
| 5      | Velocidad angular θ̇₂             | -9π          | 9π           |

**Notas:**
- θ₁: ángulo del brazo superior respecto a la vertical.
- θ₂: ángulo del brazo inferior respecto al superior.
- Usar cos(θ) y sin(θ) permite manejar la continuidad del ángulo (evita problemas al pasar de π a -π).
- Las velocidades angulares tienen límites físicos aproximados para evitar inestabilidad en simulaciones.

**Interpretación de los Estados - Acrobot**

- **cos(θ₁), sin(θ₁)** → Describe la orientación del brazo superior respecto a la vertical.  
- **cos(θ₂), sin(θ₂)** → Describe la orientación del brazo inferior respecto al brazo superior.  
- **θ̇₁** → Velocidad angular del brazo superior (qué tan rápido rota).  
- **θ̇₂** → Velocidad angular del brazo inferior (qué tan rápido rota).  

**Notas de interpretación:**
- Los valores de cos y sin permiten manejar la continuidad de los ángulos, evitando saltos abruptos al pasar de π a -π.  
- Las velocidades angulares indican la energía cinética de cada brazo y ayudan al agente a planificar cómo generar impulso para alcanzar la altura objetivo.  

In [11]:
# Obtener límites del espacio de observación
low = env.observation_space.low
high = env.observation_space.high

# Extraemos cada componente del estado (6 variables)
cos1_min, sin1_min, cos2_min, sin2_min, thdot1_min, thdot2_min = low
cos1_max, sin1_max, cos2_max, sin2_max, thdot1_max, thdot2_max = high

print("cos(θ1)           ∈ [{}, {}]".format(cos1_min, cos1_max))
print("sin(θ1)           ∈ [{}, {}]".format(sin1_min, sin1_max))
print("cos(θ2)           ∈ [{}, {}]".format(cos2_min, cos2_max))
print("sin(θ2)           ∈ [{}, {}]".format(sin2_min, sin2_max))
print("Velocidad θ̇1      ∈ [{}, {}]".format(thdot1_min, thdot1_max))
print("Velocidad θ̇2      ∈ [{}, {}]".format(thdot2_min, thdot2_max))

cos(θ1)           ∈ [-1.0, 1.0]
sin(θ1)           ∈ [-1.0, 1.0]
cos(θ2)           ∈ [-1.0, 1.0]
sin(θ2)           ∈ [-1.0, 1.0]
Velocidad θ̇1      ∈ [-12.566370964050293, 12.566370964050293]
Velocidad θ̇2      ∈ [-28.274333953857422, 28.274333953857422]


In [12]:
# Imprime la descripción completa del espacio de observaciones
# En Acrobot normalmente verás algo como:
# Box(low, high, (6,), float32)
# → Box: espacio continuo
# → (6,): 6 variables de estado
print(env.observation_space)

# Imprime los valores mínimos posibles para cada dimensión del estado
# Orden: [cos(θ1), sin(θ1), cos(θ2), sin(θ2), θ̇1, θ̇2]
print("Low:", env.observation_space.low)

# Imprime los valores máximos posibles para cada dimensión del estado
print("High:", env.observation_space.high)

# Imprime la forma del estado
# (6,) significa que cada estado tiene 6 variables
print("Shape:", env.observation_space.shape)

# Imprime el tipo de dato
# float32 → números decimales de precisión simple
print("Dtype:", env.observation_space.dtype)

Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)
Low: [ -1.        -1.        -1.        -1.       -12.566371 -28.274334]
High: [ 1.        1.        1.        1.       12.566371 28.274334]
Shape: (6,)
Dtype: float32


**Acciones (Action Space) – Valores Discretos (Acrobot)**

El espacio de acciones es **discreto** y representa los torques que puede aplicar el agente en la articulación central:

- $a \in \{0,1,2\}$

  - $0$ → Aplicar torque negativo
  - $1$ → No aplicar torque
  - $2$ → Aplicar torque positivo

**Tabla de Acciones - Acrobot**

| Acción | Descripción                     | Tipo      |
|--------|---------------------------------|----------|
| 0      | Aplicar torque negativo         | Discreta |
| 1      | No aplicar torque               | Discreta |
| 2      | Aplicar torque positivo         | Discreta |

Imprime la descripción completa del espacio de **acciones** del entorno.

Esto nos dice el tipo de espacio (Discrete, Box, etc.), los valores mínimos y máximos

In [16]:
print(env.action_space)

Discrete(3)


Verificamos si el espacio de acciones es discreto o continuo

In [18]:
# Espacios discretos representan acciones como enteros: 0, 1, 2, ...
if isinstance(env.action_space, gym.spaces.Discrete):
    # Imprime el número total de acciones posibles
    print("Número de acciones discretas:", env.action_space.n)
    # Imprime la lista de acciones posibles (enteros)
    print("Acciones posibles:", list(range(env.action_space.n)))
else:
    # Para espacios continuos (Box), imprimimos los rangos mínimos y máximos
    # Esto representa el valor mínimo y máximo que se puede aplicar para cada acción
    print("Acciones continuas - mínimo:", env.action_space.low)
    print("Acciones continuas - máximo:", env.action_space.high)
    # También podemos mostrar la forma y tipo de datos
    print("Shape del espacio de acciones:", env.action_space.shape)
    print("Tipo de datos:", env.action_space.dtype)

Número de acciones discretas: 3
Acciones posibles: [0, 1, 2]


La función  <code> discretizar </code> normaliza el estado 'valor' usando los límites del espacio de observaciones
$$\frac{\textit{valor} - \textit{mínimo}}{\textit{máximo} - \textit{mínimo}}$$
- Cada dimensión queda entre $0$ y $1$.

In [20]:
def discretizar(valor):
    aux = ((valor - env.observation_space.low) / 
           (env.observation_space.high - env.observation_space.low)) * 19
    # Multiplicamos por 20 para crear 20 intervalos discretos por dimensión

    # Convertimos los valores a enteros (truncando decimales)
    aux = aux.astype(np.int32)

    # Devolvemos una tupla de enteros
    # Esto permite usarla como índice en una tabla Q o diccionario
    return tuple(aux)

En **Acrobot-v1**, la discretización se utiliza porque **los estados del entorno son continuos**. Por ejemplo, el espacio de estados incluye:

- `cos(θ1)`, `sin(θ1)`: orientación del brazo superior  
- `cos(θ2)`, `sin(θ2)`: orientación del brazo inferior  
- `θ̇1`: velocidad angular del brazo superior  
- `θ̇2`: velocidad angular del brazo inferior  

Dado que la mayoría de estos valores son **continuos**, no es posible usar directamente una **Q-table**, ya que esto requeriría un número infinito de combinaciones de estados.  

Por ello, se recurre a:

- **Discretización del espacio de estados** (dividir rangos continuos en "bins")  
- **Aproximaciones por función** (como redes neuronales)  

Esto permite aplicar métodos basados en **Q-learning** incluso en entornos con estados continuos como Acrobot.

Al discretizar, se convierten estos valores continuos en un número finito de “intervalos” o estados. Esto permite representar el problema con una tabla finita y aplicar Q-learning de forma práctica.

Además, reduce la complejidad del problema y facilita el aprendizaje del agente, ya que agrupa estados similares. Sin embargo, existe un equilibrio: usar pocos intervalos simplifica el aprendizaje pero pierde precisión, mientras que usar muchos mejora la precisión pero incrementa el tiempo de entrenamiento y el tamaño de la Q-table.


**Estado inicial - Acrobot**

En **Acrobot-v1**, el estado inicial del brazo se asigna típicamente de manera aleatoria dentro de ciertos rangos:

- La orientación de los brazos (`θ1`, `θ2`) se inicializa generalmente cerca de `0` (brazos colgando hacia abajo) con una pequeña aleatoriedad:
  - Por ejemplo, `θ1 ∼ 𝒰(-0.1, 0.1)`  
  - Por ejemplo, `θ2 ∼ 𝒰(-0.1, 0.1)`  
- Las velocidades angulares (`θ̇1`, `θ̇2`) se asignan típicamente a `0` o a un valor pequeño cercano a cero.

**Notas:**

- Usar pequeñas variaciones aleatorias en el estado inicial ayuda al agente a generalizar mejor durante el entrenamiento.  
- No hay posiciones absolutas ni contacto con el suelo, ya que Acrobot es un sistema de brazo pendular.  
- El extremo del brazo comienza “colgando” y el agente debe balancear los brazos para alcanzar la altura objetivo.  

**Valor máximo al realizar discretización**

In [25]:
# Estado inicial máximo (todos los valores posibles en su límite superior)
estado_inicial_max = np.array([
    cos1_max,   # cos(θ1)
    sin1_max,   # sin(θ1)
    cos2_max,   # cos(θ2)
    sin2_max,   # sin(θ2)
    thdot1_max, # Velocidad angular θ̇1
    thdot2_max  # Velocidad angular θ̇2
])

estado_inicial_max

array([ 1.      ,  1.      ,  1.      ,  1.      , 12.566371, 28.274334],
      dtype=float32)

In [26]:
discretizar(estado_inicial_max) #Estado inicial

(19, 19, 19, 19, 19, 19)

**Valor aleatorio al realizar discretización**

In [28]:
estado_inicial =  env.reset()[0]
estado_inicial

array([ 0.99988633,  0.01507825,  0.999725  , -0.02345224,  0.05031167,
        0.08722378], dtype=float32)

In [29]:
discretizar(estado_inicial) #Estado inicial

(18, 9, 18, 9, 9, 9)

**Valor mínimo al realizar discretización**

In [31]:
# Estado inicial mínimo (todos los valores posibles en su límite inferior)
estado_inicial_min = np.array([
    cos1_min,   # cos(θ1)
    sin1_min,   # sin(θ1)
    cos2_min,   # cos(θ2)
    sin2_min,   # sin(θ2)
    thdot1_min, # Velocidad angular θ̇1
    thdot2_min  # Velocidad angular θ̇2
])

estado_inicial_min

array([ -1.      ,  -1.      ,  -1.      ,  -1.      , -12.566371,
       -28.274334], dtype=float32)

In [32]:
discretizar(estado_inicial_min) #Estado inicial

(0, 0, 0, 0, 0, 0)

# Modelo

En Acrobot, podemos crear una Q-table para un entorno con **6 dimensiones discretizadas** y **3 acciones posibles**.

- Cada celda `q_table[i, j, k, l, m, n, a]` representa la estimación de la recompensa esperada para el estado discreto `(i, j, k, l, m, n)` al tomar la acción `a`.

Índices:

- `i` → índice para `cos(θ1)`  
- `j` → índice para `sin(θ1)`  
- `k` → índice para `cos(θ2)`  
- `l` → índice para `sin(θ2)`  
- `m` → índice para la velocidad angular `θ̇1`  
- `n` → índice para la velocidad angular `θ̇2`  
- `a` → acción tomada (`0: torque negativo`, `1: no torque`, `2: torque positivo`)  

**Notas:**

- Cada dimensión continua del estado se discretiza en “bins” para poder usar Q-learning.  
- Esta Q-table tiene tamaño `(bins_cos1, bins_sin1, bins_cos2, bins_sin2, bins_thdot1, bins_thdot2, 3)`.  
- La Q-table permite al agente estimar la recompensa esperada de cada acción en cada estado discretizado.

Creamos la Q-table con valores iniciales aleatorios entre `-1` y `1`.  

**Dimensiones:** `[20, 20, 20, 20, 20, 20, 3]`  

- `20` divisiones para `cos(θ1)` (índices 0 a 19)  
- `20` divisiones para `sin(θ1)` (índices 0 a 19)  
- `20` divisiones para `cos(θ2)` (índices 0 a 19)  
- `20` divisiones para `sin(θ2)` (índices 0 a 19)  
- `20` divisiones para la velocidad angular `θ̇1` (índices 0 a 19)  
- `20` divisiones para la velocidad angular `θ̇2` (índices 0 a 19)  
- `3` → número de acciones posibles (`0: torque negativo`, `1: no torque`, `2: torque positivo`)  

**Notas:**  
- Cada dimensión del estado continuo se discretiza en 20 bins para poder usar Q-learning.  
- La Q-table permite estimar la recompensa esperada para cada combinación de estado discretizado y acción.  
- Esta tabla será la base para actualizar las políticas del agente durante el entrenamiento.

In [36]:
q_table = np.random.uniform(low=-1, high=1, size=[20,20,20,20,20,20, 3])

In [37]:
len(q_table)

20

## Entrenamiento

## Parámetros de Q-learning

In [40]:
alfa = 0.3        # Tasa de aprendizaje: qué tanto se actualiza la Q-table en cada iteración
gamma = 0.95      # Factor de descuento: qué tanto importan las recompensas futuras
episodios = 100000  # Número total de episodios de entrenamiento
epsilon = 2       # Parámetro de exploración (controla aleatoriedad en acciones)
lista_recompenzas = [] # Lista para guardar la recompensa total obtenida en cada episodio

## Bucle principal de entrenamiento

In [42]:
for episodio in range(episodios):

    # Reiniciamos el entorno y discretizamos el estado inicial
    estado = discretizar(env.reset()[0])

    # Variables de control del episodio
    final = False
    f1 = False
    f2 = False
    
    recompensa_total = 0

    # ==============================
    # INTERACCIÓN CON EL ENTORNO
    # ==============================
    while not final:

        # Política ε-greedy
        valor_random = randint(0,10)
        #print(f"\nEpisodio: {episodio} | Valor random: {valor_random}")

        if valor_random > epsilon:
            accion = np.argmax(q_table[estado])
            tipo = "EXPLOTACIÓN"
        else:
            accion = randint(0,2) #{0,1,2}
            tipo = "EXPLORACIÓN"

        #print(f"Tipo: {tipo}, Acción: {accion}, Estado actual: {estado}")

        # Ejecutamos la acción
        nuevo_estado, recompensa, f1, f2, info = env.step(accion)

        #print(f"nuevo_estado: {nuevo_estado}, recompensa: {recompensa}, f1: {f1}, f2: {f2}, info: {info}")

        # ==============================
        # ACTUALIZACIÓN Q-LEARNING
        # ==============================
        estado_discreto_nuevo = discretizar(nuevo_estado)

        valor_actual = q_table[estado][accion]
        max_futuro = np.max(q_table[estado_discreto_nuevo])

        #print(f"Q actual: {valor_actual}, max Q futuro: {max_futuro}")

        # Fórmula Q-learning
        q_table[estado][accion] = (
            valor_actual +
            alfa * (recompensa + gamma * max_futuro - valor_actual)
        )

        #print(f"Q actualizado: {q_table[estado][accion]}")

        # Actualizamos estado
        estado = estado_discreto_nuevo

        # Acumulamos recompensa
        recompensa_total += recompensa

        # Verificamos fin
        final = f1 or f2

        #print(f"Recompensa acumulada: {recompensa_total}, Final: {final}")
        #print("--------------------------------------------------")

    # Guardamos recompensa del episodio
    lista_recompenzas.append(recompensa_total)

    # Print cada 100 episodios
    if (episodio + 1) % 1000 == 0:
        print("Episodio: " + str(episodio) +
              " | Recompensa promedio: " + str(np.mean(lista_recompenzas)))

# Cerramos entorno
env.close()

Episodio: 999 | Recompensa promedio: -482.932
Episodio: 1999 | Recompensa promedio: -472.011
Episodio: 2999 | Recompensa promedio: -460.94233333333335
Episodio: 3999 | Recompensa promedio: -452.31075
Episodio: 4999 | Recompensa promedio: -445.456
Episodio: 5999 | Recompensa promedio: -440.0355
Episodio: 6999 | Recompensa promedio: -433.4242857142857
Episodio: 7999 | Recompensa promedio: -427.7205
Episodio: 8999 | Recompensa promedio: -422.3644444444444
Episodio: 9999 | Recompensa promedio: -418.5769
Episodio: 10999 | Recompensa promedio: -415.1947272727273
Episodio: 11999 | Recompensa promedio: -412.0346666666667
Episodio: 12999 | Recompensa promedio: -408.4952307692308
Episodio: 13999 | Recompensa promedio: -404.9147857142857
Episodio: 14999 | Recompensa promedio: -401.85853333333336
Episodio: 15999 | Recompensa promedio: -399.07275
Episodio: 16999 | Recompensa promedio: -396.3983529411765
Episodio: 17999 | Recompensa promedio: -393.993
Episodio: 18999 | Recompensa promedio: -391.5138

Crear el entorno de Acrobot
- <code> render_mode="human" </code> permite ver la animación en pantalla 

In [69]:
env = gym.make("Acrobot-v1", render_mode="human")

In [71]:
print("Estado:", estado, "Tipo:", type(estado))
print("q_table shape:", q_table.shape)
print("q_table[estado]:", q_table[estado])

Estado: (0, 10, 12, 18, 10, 8) Tipo: <class 'tuple'>
q_table shape: (20, 20, 20, 20, 20, 20, 3)
q_table[estado]: [0.00151075 0.24799821 0.15719262]


In [73]:
print(f"Estado: {estado}, len: {len(estado)}")

Estado: (0, 10, 12, 18, 10, 8), len: 6


In [75]:
# Reinicia el entorno y obtiene el estado inicial (continuo)
# Luego se discretiza para poder usar la Q-table
estado = discretizar(env.reset()[0])

# Variables de control del episodio
final = False   # Indica si el episodio terminó
f1 = False      # Bandera de éxito (llegó a la meta)
f2 = False      # Bandera de fallo (se terminó el tiempo)


# Mientras el episodio no termine
while not final:
    
    # Elige la mejor acción según la Q-table (explotación)
    accion = np.argmax(q_table[estado])
      
    # Ejecuta la acción en el entorno
    # Devuelve el nuevo estado, la recompensa y si terminó el episodio
    nuevo_estado, recompensa, f1, f2, info = env.step(accion)

    # Convierte el nuevo estado continuo a discreto
    estado = discretizar(nuevo_estado)
  
    # Verifica si el episodio terminó (éxito o fallo)
    final = f1 or f2

time.sleep(10)
# Cierra el entorno
env.close()